# 校园快递包裹分类识别 —— Kaggle 端到端训练 Notebook

**使用前，务必先在右侧 Settings 面板里做两件事：**
1. **Accelerator** 选 GPU（T4 x2 或 P100）
2. **Internet** 打开（默认是关的，不打开没法 `git clone` 也没法 `pip install`）

流程 11 步（比 Colab 版少了 Drive 挂载相关步骤，Kaggle 不需要）：

1. 从 GitHub 克隆项目
2. 检查 GPU
3. 安装依赖
4. 检查数据是否已就位
5. 整理公开数据集为统一 YOLO 格式
6. 配置类别映射
7. 合并 + 清洗 + 划分 + 增强
8. 阶段1训练（公开数据集预训练）
9. 阶段2训练（校园数据集迁移微调）
10. 评估
11.（可选）启动推理服务测试

**使用方法：** 改好步骤1的仓库地址和分支名，从上到下依次运行到底即可。已经做完的步骤会自动检测并跳过。

**注意 Kaggle 的两个特性：**
- `/kaggle/working` 在**当前 session 内**是持久的，但换一个新 session（比如重启 Notebook）默认还是空的，
  除非你之前点过 "Save Version"（Save & Run All）把输出提交保存下来
- 免费账号每周有 **30 小时 GPU 额度**，跑完记得在右上角关掉 Session，不然会一直计时


## 步骤1：从 GitHub 克隆项目

In [1]:
import os

REPO_URL = "https://github.com/cui0122/campus-express-yolov8.git"   # ← 改成你的实际仓库地址
BRANCH = "master"    # ← 项目在哪个分支就填哪个，比如 master / main / dev
CLONE_DIR = "/content/campus-express-yolov8"

if os.path.exists(os.path.join(CLONE_DIR, ".git")):
    print(f"📁 已存在克隆目录，切换到 {BRANCH} 分支并 pull 更新: {CLONE_DIR}")
    !cd "{CLONE_DIR}" && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f"📥 克隆仓库（分支: {BRANCH}）到: {CLONE_DIR}")
    !git clone -b {BRANCH} --single-branch "{REPO_URL}" "{CLONE_DIR}"

print("\n当前分支:")
!cd "{CLONE_DIR}" && git branch --show-current

print("\n目录内容:")
!ls "{CLONE_DIR}"
  # ← 改成你的实际仓库地址
BRANCH = "master"    # ← 项目在哪个分支就填哪个
CLONE_DIR = "/kaggle/working/campus-express-yolov8"

if os.path.exists(os.path.join(CLONE_DIR, ".git")):
    print(f"📁 已存在克隆目录，切换到 {BRANCH} 分支并 pull 更新: {CLONE_DIR}")
    !cd "{CLONE_DIR}" && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f"📥 克隆仓库（分支: {BRANCH}）到: {CLONE_DIR}")
    !git clone -b {BRANCH} --single-branch "{REPO_URL}" "{CLONE_DIR}"

os.chdir(CLONE_DIR)
print("\n当前工作目录:", os.getcwd())
!ls


📥 克隆仓库（分支: master）到: /content/campus-express-yolov8
Cloning into '/content/campus-express-yolov8'...
remote: Enumerating objects: 12884, done.
remote: Counting objects: 100% (1982/1982), done.
remote: Compressing objects: 100% (1092/1092), done.
remote: Total 12884 (delta 6), reused 1970 (delta 4), pack-reused 10902 (from 1)
Receiving objects: 100% (12884/12884), 806.49 MiB | 41.98 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Updating files: 100% (13674/13674), done.

当前分支:
master

目录内容:
campus_train_auto.ipynb  docs	    requirements.txt  system
data			 README.md  runs	      training
📥 克隆仓库（分支: master）到: /kaggle/working/campus-express-yolov8
Cloning into '/kaggle/working/campus-express-yolov8'...
remote: Enumerating objects: 12884, done.
remote: Counting objects: 100% (1982/1982), done.
remote: Compressing objects: 100% (1092/1092), done.
remote: Total 12884 (delta 6), reused 1970 (delta 4), pack-reused 10902 (from 1)
Receiving objects: 100% (12884/12884), 806.49 MiB | 43.73 MiB/

## 训练前置状态总览

检查 `data/dataset/data.yaml` 是否已经存在（意味着合并/清洗/划分/增强全部跑完了）。
存在的话，下面第4~7步会自动全部跳过，直接从「阶段1训练」开始运行即可。


In [2]:
READY_FOR_TRAINING = os.path.exists("data/dataset/data.yaml")

if READY_FOR_TRAINING:
    print("✅ 检测到 data/dataset/data.yaml 已存在，数据集已经准备完毕。")
    print("   下面第4~7步会自动跳过，可以直接运行到「阶段1训练」那一步开始训练。")
    !cat data/dataset/data.yaml
else:
    print("ℹ️ 未检测到现成的 data/dataset/data.yaml，将按正常流程走：检查数据 -> 整理 -> 类别映射 -> 合并清洗划分增强。")


ℹ️ 未检测到现成的 data/dataset/data.yaml，将按正常流程走：检查数据 -> 整理 -> 类别映射 -> 合并清洗划分增强。


## 步骤2：检查 GPU

In [3]:
!nvidia-smi


Sun Aug 16 04:25:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
from collections import Counter
from glob import glob

def count_labels_in_split(split_dir):
    """
    统计某个数据划分目录下所有标签文件中各类别的实例数。
    返回 Counter 对象，key 为类别 ID（字符串），value 为出现次数。
    """
    counter = Counter()
    label_pattern = os.path.join(split_dir, "*.txt")
    for label_path in glob(label_pattern):
        try:
            with open(label_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        cls_id = parts[0]   # 第一个字段是类别ID
                        counter[cls_id] += 1
        except (FileNotFoundError, PermissionError):
            continue
    return counter

def main():
    # ★关键修正：要统计的是划分完的最终数据集 data/dataset，
    # 且标签文件在 labels/<split>/ 子目录下，不是直接在 <split>/ 下
    base_dir = "data/dataset"
    label_root = os.path.join(base_dir, "labels")
    # ★关键修正：split_dataset.py 生成的目录名是 train / val / test（不是 valid）
    splits = ["train", "val", "test"]

    for split in splits:
        split_dir = os.path.join(label_root, split)
        if not os.path.isdir(split_dir):
            print(f"== {split} == (目录不存在: {split_dir}，跳过)")
            continue

        counter = count_labels_in_split(split_dir)
        print(f"== {split} ==")
        if not counter:
            print("  (无标签文件或无法读取)")
        else:
            for cls_id, count in sorted(counter.items(), key=lambda x: int(x[0]) if x[0].isdigit() else x[0]):
                print(f"{count:>6} {cls_id}")
        print()

if __name__ == "__main__":
    main()

== train ==
  3495 0

== val ==
   957 0

== test ==
   492 0



## 步骤3：安装依赖

In [4]:
!pip install -q -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.8 MB/s eta 0:00:00


## 步骤4：检查数据是否已就位

如果数据集是通过 `+ Add Data` 添加的 Kaggle Dataset（只读，挂载在 `/kaggle/input/xxx`），
需要先把它复制/软链到项目的 `data/public_raw`（或 `data/campus_raw`），下面 cell 里有示例，按需取消注释。


In [5]:
# 如果公开数据集是通过 Kaggle Dataset 添加的，取消下面注释并改成你实际的输入路径：
# !mkdir -p data/public_raw
# !cp -r /kaggle/input/你的数据集名称/* data/public_raw/

# 如果校园自采数据集也是通过 Kaggle Dataset 添加的：
# !rm -rf data/campus_raw
# !cp -r /kaggle/input/你的校园数据集名称 data/campus_raw

def check(path, desc):
    ok = os.path.exists(path)
    mark = "✅" if ok else "❌"
    print(f"{mark} {desc}: {path}")
    return ok

has_public_yolo = check("data/public_yolo/images", "公开数据集(已整理为YOLO格式)")
has_public_raw = check("data/public_raw", "公开数据集(原始Roboflow导出，待整理)")
has_campus_img = check("data/campus_raw/images", "校园数据集图像目录")
has_campus_lbl = check("data/campus_raw/labels", "校园数据集标注目录")

if has_public_yolo:
    print("\n📦 检测到已整理好的公开数据集，下一步会自动跳过整理，直接进入合并。")
    SKIP_PREPARE = True
elif has_public_raw:
    print("\n📦 检测到原始 Roboflow 导出，下一步会自动整理成统一 YOLO 格式。")
    SKIP_PREPARE = False
else:
    print("\n⚠️ 既没找到 data/public_yolo 也没找到 data/public_raw，"
          "请检查仓库/Dataset 里是否已经打包了数据集，或者数据放在了别的路径下。")
    SKIP_PREPARE = None

if not (has_campus_img and has_campus_lbl):
    print("ℹ️ 未检测到校园自采数据，后续合并会自动只用公开数据集（这是正常情况，不会报错）。")


❌ 公开数据集(已整理为YOLO格式): data/public_yolo/images
✅ 公开数据集(原始Roboflow导出，待整理): data/public_raw
✅ 校园数据集图像目录: data/campus_raw/images
✅ 校园数据集标注目录: data/campus_raw/labels

📦 检测到原始 Roboflow 导出，下一步会自动整理成统一 YOLO 格式。


## 步骤5：整理公开数据集为统一 YOLO 格式（自动判断是否需要跑）

In [6]:
if READY_FOR_TRAINING:
    print("✅ 数据集已就绪，跳过整理这一步。")
elif SKIP_PREPARE is None:
    raise SystemExit("没检测到任何公开数据集，请先确认数据来源后再继续。")
elif SKIP_PREPARE:
    print("✅ data/public_yolo 已存在，跳过整理这一步。")
    !ls data/public_yolo
else:
    !python data/scripts/prepare_public_dataset.py --src data/public_raw --out data/public_yolo


检测到类别: ['Boxes', 'Parcel', 'good-parcel', 'label', 'package', 'parcel']
[OK] 共整理 3898 张图像 -> data/public_yolo
下一步：打开 data/scripts/merge_datasets.py，按上面打印出的类别名配置 CLASS_MAP。


## 步骤6：配置类别映射（数据集已就绪时自动跳过）

运行本 cell 前请先核对上一步打印出的实际类别名，按需修改下面的 `PUBLIC_CLASS_MAP`。


In [7]:
if READY_FOR_TRAINING:
    print("✅ 数据集已就绪，跳过类别映射配置这一步。")
else:
    import re

    FINAL_CLASSES = ["纸箱", "塑料袋", "泡沫箱"]

    PUBLIC_CLASS_MAP = {
    "box": "纸箱",
    "boxes": "纸箱",
    "cardboard": "纸箱",
    "cardboard box": "纸箱",
    "corrugated carton": "纸箱",
    "package": "纸箱",
    "parcel": "纸箱",
    "plastic bag": "塑料袋",
    "single-use carrier bag": "塑料袋",
    "polypropylene bag": "塑料袋",
    "courier bag": "塑料袋",
    "poly mailer": "塑料袋",
    "foam food container": "泡沫箱",
    "styrofoam": "泡沫箱",
    "styrofam piece":"泡沫箱",
    "foam box": "泡沫箱",
    "label": None,
    "person": None,
}

    classes_repr = "[" + ", ".join(f'"{c}"' for c in FINAL_CLASSES) + "]"
    map_lines = "\n".join(
        f'    "{k}": {repr(v) if v is None else chr(34)+v+chr(34)},' for k, v in PUBLIC_CLASS_MAP.items()
    )
    patch = f'FINAL_CLASSES = {classes_repr}\n\nPUBLIC_CLASS_MAP = {{\n{map_lines}\n}}'

    with open("data/scripts/merge_datasets.py", "r", encoding="utf-8") as f:
        content = f.read()

    pattern = re.compile(r'FINAL_CLASSES = \[.*?\n\}', re.S)
    new_content, n = pattern.subn(patch, content, count=1)
    assert n == 1, "没匹配到原有的 FINAL_CLASSES/PUBLIC_CLASS_MAP 段落，请检查 merge_datasets.py 是否被手动改动过"

    with open("data/scripts/merge_datasets.py", "w", encoding="utf-8") as f:
        f.write(new_content)

    print("✅ 类别映射已写入 data/scripts/merge_datasets.py")
    !sed -n '1,20p' data/scripts/merge_datasets.py


✅ 类别映射已写入 data/scripts/merge_datasets.py
"""
合并「公开数据集（YOLO格式）」+「校园自采数据集（YOLO格式）」，统一类别映射。

★★★ 使用前必须修改下面的 CLASS_MAP ★★★
key 是原始数据集里出现的类别名（不区分大小写），value 是你项目最终想用的类别名。
把不需要的类别映射成 None，脚本会自动丢弃这些框（如果一张图的所有框都被丢弃，
这张图会被跳过，除非 KEEP_NEGATIVE_IMAGES=True）。

最终类别顺序由 FINAL_CLASSES 决定，训练用的 class id 以这个列表的下标为准。
"""
import os
import shutil

# ---------------- 配置区，按你的实际情况修改 ----------------

FINAL_CLASSES = ["纸箱", "塑料袋", "泡沫箱"]

PUBLIC_CLASS_MAP = {
    "box": "纸箱",
    "boxes": "纸箱",


## 步骤7：合并 + 清洗 + 划分 + 增强（数据集已就绪时自动跳过）

In [8]:
if READY_FOR_TRAINING:
    print("✅ 数据集已就绪，跳过合并/清洗/划分/增强这一步，直接看下面的划分结果：")
    !ls data/dataset/images
else:
    !python data/scripts/merge_datasets.py
    !python data/scripts/clean_dataset.py
    !python data/scripts/split_dataset.py
    !python data/scripts/augment.py


[pub] 保留 3002 张，跳过 896 张
[campus] 保留 1907 张，跳过 1012 张
[OK] 合并完成 -> data/merged（下一步跑 clean_dataset.py）
[OK] 清洗完成。剩余 4752 张图像。
  剔除损坏图像: 0
  剔除重复图像: 157
  剔除空标注: 0
train: 3326 张
val: 950 张
test: 476 张
[OK] 划分完成，data.yaml 已生成: data/dataset/data.yaml
待增强图像数: 3326
数据增强中: 100%|███████████████████████████| 3326/3326 [03:03<00:00, 18.10张/s]
[OK] 增强完成，train 集图像总数: 9864


## 步骤8：阶段1训练（公开数据集预训练，COCO权重起步）

In [ ]:
!python training/train.py --stage 1 --config training/configs/stage1_public_pretrain.yaml


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[Stage 1] 使用初始权重: yolov8n.pt
[Stage 1] 数据集配置: data/dataset/data.yaml
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epo

## 步骤9：评估（生成论文第六章可用的指标表）

In [ ]:
!python training/eval.py --weights runs/detect/training/runs/stage2/weights/best.pt --data data/dataset/data.yaml
import pandas as pd
pd.read_csv("training/eval_report.csv")


## 步骤10（可选）：启动推理服务本地测试

需要先把 `training/runs/stage2/weights/best.pt` 拷贝到 `system/backend/models/best.pt`。
Kaggle 里没有公网入口，这一步主要用来在 Notebook 内部自测接口逻辑是否正常。


In [ ]:
!cp runs/detect/training/runs/stage2/weights/best.pt system/backend/models/best.pt
import subprocess, time
proc = subprocess.Popen(
    ["uvicorn", "main:app", "--reload" ,"--port", "8000"],
    cwd="system/backend"
)
time.sleep(3)
print("推理服务已启动: http://127.0.0.1:8000/detect")
print("停止服务请运行: proc.terminate()")


## （可选）打包 runs 目录，方便下载权重/日志/曲线图

Kaggle 右侧 Output 面板可以直接下载 `/kaggle/working` 下的文件，
打包成一个 zip 更方便一次性下载。


In [ ]:
!zip -r -q /kaggle/working/training_runs.zip runs/detect/training/runs
print("已生成 /kaggle/working/training_runs.zip，可在右侧 Output 面板下载")
